# Scamless - Training Run (GPU)

Runtime > Change runtime type > **T4 GPU**, then Run all.
Clones the repo, builds the dataset, fine-tunes, exports int8 ONNX, evals, gates, saves artifacts to Google Drive.

In [ ]:
%cd /content
import pathlib, shutil

if pathlib.Path("/content/scamless").exists():
    %cd /content/scamless
    !git fetch origin
    !git reset --hard origin/main
else:
    !git clone https://github.com/RishavGupta01/Scamless.git /content/scamless
    %cd /content/scamless
!pip install -q -e "training[dev]"
import sys
sys.path.insert(0, "/content/scamless/training/src")

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())

In [ ]:
from google.colab import drive
import pathlib

drive.mount("/content/drive")
DRIVE_ART = pathlib.Path("/content/drive/MyDrive/scamless/artifacts")
DRIVE_ART.mkdir(parents=True, exist_ok=True)
print("artifacts will persist to", DRIVE_ART)

In [ ]:
!python -m scamless.pipeline

In [ ]:
import json
import pathlib
import sys
sys.path.insert(0, "/content/scamless/training/src")

import pandas as pd

from scamless.eval.baseline import heuristic_predict
from scamless.eval.harness import compute_metrics

# heuristic baseline for comparison (does NOT overwrite metrics.json)
test_df = pd.read_parquet("training/data/processed/messages_test.parquet")
base = compute_metrics(test_df, [heuristic_predict(str(t)) for t in test_df["text"]])
print("baseline macro_f1:", base["macro_f1"], "fp_rate:", base["false_positive_rate"])
print("trained:", json.loads(open("artifacts/eval/metrics.json").read()))

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")
shutil.copytree("artifacts", "/content/drive/MyDrive/scamless/artifacts", dirs_exist_ok=True)
print("saved to", "/content/drive/MyDrive/scamless/artifacts")